# Binary SwinV2-Tiny Dermoscope-Only — 5-Fold Cross-Validation (Full Fine-Tune)

K-Fold CV version of the full fine-tuning experiment. Instead of a single train/val split, we train 5 independent models and average their results for more robust performance estimates (mean ± std).

**Protocol:** Hold out 10% test set → 5-fold CV on remaining 90% → each fold: warmup (frozen, 5 epochs) + full fine-tune entire backbone (up to 30 epochs) → evaluate all folds on shared test set → ensemble by averaging probabilities.

**Binary labels:**
| Original Class | Binary Label |
|---|---|
| Melanoma | **Malignant** (1) |
| BCC | **Malignant** (1) |
| SCC | **Malignant** (1) |
| Actinic Keratosis | **Malignant** (1) |
| Malignant_Other | **Malignant** (1) |
| Melanocytic_Nevus | **Benign** (0) |
| Seborrheic_Keratosis | **Benign** (0) |
| Dermatofibroma | **Benign** (0) |
| Hemangioma | **Benign** (0) |
| Fibrous_Papule | **Benign** (0) |
| Other_Benign | **Benign** (0) |


## 1. Setup & Imports

In [ ]:
import os
import copy
import random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, precision_recall_fscore_support, fbeta_score
)

from transformers import Swinv2ForImageClassification

from utils import (
    BinaryDermoscopeSkinDataset,
    get_transforms,
    BINARY_MAP,
    CLASS_NAMES,
    BinaryFocalLossWithSmoothing,
    mixup_data,
    mixup_criterion,
    find_optimal_threshold,
    compute_metrics,
    MIN_SENSITIVITY_TARGET
)

print("Imports complete.")

In [ ]:
# ── Configuration ──
IMAGES_DIR = "../DataCleaning/Images"
MANIFEST_PATH = "../DataCleaning/instances.csv"
BATCH_SIZE = 12
SEED = 42
K_FOLDS = 5

TRAIN_THRESHOLD = 0.5

# Training config
WARMUP_EPOCHS = 5
FT_EPOCHS = 30
PATIENCE = 8
MIXUP_ALPHA = 0.15

LR_BACKBONE = 2e-5
LR_HEAD = 1e-4
LABEL_SMOOTHING = 0.05

NUM_WORKERS = 0 if os.name == "nt" else min(os.cpu_count() or 8, 12)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

print(f"Using: {DEVICE}")
print(f"Mixed precision: {USE_AMP}")
print(f"PyTorch: {torch.__version__}")
print(f"\n🧪 STRATIFIED {K_FOLDS}-FOLD CV — FULL BACKBONE FINE-TUNE")
print(f"  • Phase 2 unfreezes ENTIRE backbone")
print(f"  • {K_FOLDS} folds, each with warmup + fine-tuning")
print(f"  • Reports mean ± std of all metrics across folds")

if torch.cuda.is_available():
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Transforms (256x256 for SwinV2-Tiny)

In [ ]:
# SwinV2-Tiny uses 256x256 input size
dscope_train_transform, eval_transform = get_transforms(image_size=256, resize_size=288)

print("✓ Dermoscope transforms created (256x256 for SwinV2-Tiny)")

## 3. Load Data & Hold Out Test Set

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
print(f"Total instances: {len(manifest)}")
print(f"\nOriginal class distribution:")
print(manifest["cancer_type"].value_counts())

manifest["binary_label"] = manifest["cancer_type"].map(BINARY_MAP)
print(f"\nBinary distribution:")
for lbl, name in enumerate(CLASS_NAMES):
    count = (manifest["binary_label"] == lbl).sum()
    pct = 100 * count / len(manifest)
    print(f"  {name} ({lbl}): {count} ({pct:.1f}%)")

# Hold out 10% test set (IDENTICAL to baseline)
sss_trainval_test = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)
trainval_idx, test_idx = next(sss_trainval_test.split(manifest, manifest["binary_label"]))

trainval_manifest = manifest.iloc[trainval_idx].reset_index(drop=True)
test_manifest = manifest.iloc[test_idx].reset_index(drop=True)

print(f"\nHeld-out test set: {len(test_manifest)} instances")
print(f"Remaining for K-Fold CV: {len(trainval_manifest)} instances")

test_dataset = BinaryDermoscopeSkinDataset(
    test_manifest, IMAGES_DIR, transform=eval_transform
)
loader_kwargs = {'num_workers': NUM_WORKERS, 'pin_memory': True} if torch.cuda.is_available() else {}
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
print(f"Test dataset samples: {len(test_dataset)}")

## 4. Model Architecture & Utilities

In [ ]:
class BinarySwinV2Model(nn.Module):
    """
    SwinV2-Tiny backbone with custom classification head.
    Full backbone unfrozen in Phase 2.
    """
    def __init__(self, backbone, embed_dim=768, hidden=512, dropout=0.3):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1)
        )

    def forward(self, x):
        # Get pooled output from SwinV2 (last_hidden_state pooled)
        outputs = self.backbone.swinv2(x)
        # Use the pooled output (mean of last hidden state)
        pooled = outputs.last_hidden_state.mean(dim=1)
        logit = self.classifier(pooled)
        return logit.squeeze(-1)


def unfreeze_entire_backbone(model):
    """Unfreeze entire SwinV2 backbone for full fine-tuning."""
    for param in model.backbone.parameters():
        param.requires_grad = True


print("✓ Model and utility functions defined.")

## 5. Load Pretrained SwinV2-Tiny Backbone (Reference Copy)

In [ ]:
# Load once — we'll deep copy this for each fold so every fold starts from pretrained weights
swinv2_backbone_ref = Swinv2ForImageClassification.from_pretrained(
    "microsoft/swinv2-tiny-patch4-window8-256",
    num_labels=1,
    ignore_mismatched_sizes=True
)
embed_dim = swinv2_backbone_ref.config.hidden_size  # 768 for SwinV2-Tiny
print(f"SwinV2-Tiny loaded. Embed dim: {embed_dim}")
print("This backbone will be deep-copied for each fold.")

## 6. Stratified 5-Fold Cross-Validation Training Loop

Each fold:
1. **Phase 1** — Warmup (frozen backbone, train head only, 5 epochs)
2. **Phase 2** — Full fine-tune entire backbone (up to 30 epochs, early stopping on val AUC-ROC)
3. **Evaluate** on both the fold's validation set and the shared held-out test set

In [ ]:
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

# Storage for per-fold results
fold_val_results = []   # validation fold metrics
fold_test_results = []  # test set metrics (each fold's best model evaluated on held-out test)
fold_histories = []     # training curves per fold
fold_thresholds = []    # optimal thresholds per fold
fold_test_probs = []    # test set probabilities per fold (for ensemble later)

print(f"\n{'='*70}")
print(f"  STRATIFIED {K_FOLDS}-FOLD CROSS-VALIDATION")
print(f"{'='*70}\n")

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(trainval_manifest, trainval_manifest["binary_label"])):
    print(f"\n{'#'*70}")
    print(f"  FOLD {fold_idx + 1} / {K_FOLDS}")
    print(f"{'#'*70}")

    # ── Seed reset for reproducibility within each fold ──
    torch.manual_seed(SEED + fold_idx)
    np.random.seed(SEED + fold_idx)
    random.seed(SEED + fold_idx)

    # ── Split ──
    fold_train_manifest = trainval_manifest.iloc[train_idx].reset_index(drop=True)
    fold_val_manifest = trainval_manifest.iloc[val_idx].reset_index(drop=True)
    print(f"  Train: {len(fold_train_manifest)} | Val: {len(fold_val_manifest)}")

    for lbl, name in enumerate(CLASS_NAMES):
        tr_cnt = (fold_train_manifest["binary_label"] == lbl).sum()
        va_cnt = (fold_val_manifest["binary_label"] == lbl).sum()
        print(f"    {name}: train={tr_cnt}, val={va_cnt}")

    # ── Datasets & loaders ──
    fold_train_dataset = BinaryDermoscopeSkinDataset(
        fold_train_manifest, IMAGES_DIR, transform=dscope_train_transform
    )
    fold_val_dataset = BinaryDermoscopeSkinDataset(
        fold_val_manifest, IMAGES_DIR, transform=eval_transform
    )

    train_labels = [label for _, label in fold_train_dataset]
    class_counts = {0: train_labels.count(0), 1: train_labels.count(1)}
    total_train = len(train_labels)
    class_weights = {cls: total_train / count for cls, count in class_counts.items()}
    sample_weights = torch.tensor([class_weights[label] for label in train_labels], dtype=torch.float)
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    fold_train_loader = DataLoader(fold_train_dataset, batch_size=BATCH_SIZE, sampler=sampler, **loader_kwargs)
    fold_val_loader = DataLoader(fold_val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

    print(f"  Dataset samples — Train: {len(fold_train_dataset)} | Val: {len(fold_val_dataset)}")

    # ── Fresh model from pretrained backbone ──
    backbone_copy = copy.deepcopy(swinv2_backbone_ref)
    model = BinarySwinV2Model(
        backbone=backbone_copy, embed_dim=embed_dim, hidden=512, dropout=0.3
    ).to(DEVICE)

    # ── Pos weight for loss ──
    pos_weight = torch.tensor([class_counts[0] / class_counts[1]]).to(DEVICE)
    criterion = BinaryFocalLossWithSmoothing(gamma=2.0, pos_weight=pos_weight, label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler('cuda') if USE_AMP else None

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 1: Warmup (Frozen Backbone)
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n  ── Phase 1: Warmup ({WARMUP_EPOCHS} epochs, frozen backbone) ──")
    for param in model.backbone.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

    optimizer_warmup = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler_warmup = CosineAnnealingLR(optimizer_warmup, T_max=WARMUP_EPOCHS)

    for epoch in range(WARMUP_EPOCHS):
        model.train()
        train_loss, correct, total = 0.0, 0, 0
        for imgs, labels in fold_train_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True).float()
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    logits = model(imgs)
                    loss = criterion(logits, labels)
                optimizer_warmup.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer_warmup)
                scaler.update()
            else:
                logits = model(imgs)
                loss = criterion(logits, labels)
                optimizer_warmup.zero_grad()
                loss.backward()
                optimizer_warmup.step()
            train_loss += loss.item()
            preds = (torch.sigmoid(logits) >= TRAIN_THRESHOLD).long()
            total += labels.size(0)
            correct += preds.eq(labels.long()).sum().item()
        scheduler_warmup.step()
        avg_loss = train_loss / len(fold_train_loader)
        acc = 100. * correct / total
        print(f"    Warmup Epoch [{epoch+1}/{WARMUP_EPOCHS}] Loss: {avg_loss:.4f} | Acc: {acc:.2f}%")

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 2: Fine-Tune Entire Backbone (Early Stopping on AUC-ROC)
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n  ── Phase 2: Fine-tuning entire backbone ({FT_EPOCHS} max epochs) ──")
    unfreeze_entire_backbone(model)
    for param in model.classifier.parameters():
        param.requires_grad = True

    optimizer_ft = torch.optim.AdamW([
        {'params': [p for p in model.backbone.parameters() if p.requires_grad], 'lr': LR_BACKBONE},
        {'params': model.classifier.parameters(), 'lr': LR_HEAD},
    ], weight_decay=1e-4)
    scheduler_ft = CosineAnnealingLR(optimizer_ft, T_max=FT_EPOCHS)

    best_val_auc = 0.0
    best_model_state = None
    epochs_no_improve = 0
    ft_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [],
                  'val_auc': [], 'val_sens': [], 'val_spec': []}

    for epoch in range(FT_EPOCHS):
        model.train()
        train_loss, correct, total = 0.0, 0, 0
        for imgs, labels in fold_train_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True).float()
            if MIXUP_ALPHA > 0 and np.random.rand() < 0.5:
                imgs, labels_a, labels_b, lam = mixup_data(imgs, labels, alpha=MIXUP_ALPHA)
                use_mixup = True
            else:
                use_mixup = False
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    logits = model(imgs)
                    if use_mixup:
                        loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)
                    else:
                        loss = criterion(logits, labels)
                optimizer_ft.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer_ft)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer_ft)
                scaler.update()
            else:
                logits = model(imgs)
                if use_mixup:
                    loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)
                else:
                    loss = criterion(logits, labels)
                optimizer_ft.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer_ft.step()
            train_loss += loss.item()
            preds = (torch.sigmoid(logits) >= TRAIN_THRESHOLD).long()
            if use_mixup:
                total += labels_a.size(0)
                correct += (lam * preds.eq(labels_a.long()).float() +
                           (1 - lam) * preds.eq(labels_b.long()).float()).sum().item()
            else:
                total += labels.size(0)
                correct += preds.eq(labels.long()).sum().item()
        scheduler_ft.step()
        avg_train_loss = train_loss / len(fold_train_loader)
        train_acc = 100. * correct / total

        # Validation
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        val_preds, val_true, val_probs = [], [], []
        with torch.no_grad():
            for imgs, labels in fold_val_loader:
                imgs = imgs.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True).float()
                if USE_AMP:
                    with torch.amp.autocast('cuda'):
                        logits = model(imgs)
                        loss = criterion(logits, labels)
                else:
                    logits = model(imgs)
                    loss = criterion(logits, labels)
                val_loss += loss.item()
                probs = torch.sigmoid(logits)
                preds = (probs >= TRAIN_THRESHOLD).long()
                total += labels.size(0)
                correct += preds.eq(labels.long()).sum().item()
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(labels.long().cpu().numpy())
                val_probs.extend(probs.cpu().numpy())
        avg_val_loss = val_loss / len(fold_val_loader)
        val_acc = 100. * correct / total
        val_auc = roc_auc_score(val_true, val_probs)
        tn, fp, fn, tp = confusion_matrix(val_true, val_preds).ravel()
        val_sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        val_spec = tn / (tn + fp) if (tn + fp) > 0 else 0

        ft_history['train_loss'].append(avg_train_loss)
        ft_history['train_acc'].append(train_acc)
        ft_history['val_loss'].append(avg_val_loss)
        ft_history['val_acc'].append(val_acc)
        ft_history['val_auc'].append(val_auc)
        ft_history['val_sens'].append(val_sens)
        ft_history['val_spec'].append(val_spec)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
            marker = " ★"
        else:
            epochs_no_improve += 1
            marker = ""

        print(f"    FT Epoch [{epoch+1}/{FT_EPOCHS}] Train: {avg_train_loss:.4f}/{train_acc:.1f}% | "
              f"Val: {avg_val_loss:.4f}/{val_acc:.1f}% | AUC: {val_auc:.4f} | "
              f"Sens: {val_sens:.4f} | Spec: {val_spec:.4f}{marker}")

        if epochs_no_improve >= PATIENCE:
            print(f"    Early stopping after {PATIENCE} epochs without AUC improvement.")
            break

    fold_histories.append(ft_history)

    # ── Load best model for this fold ──
    model.load_state_dict(best_model_state)
    print(f"\n  ✓ Fold {fold_idx+1} best val AUC: {best_val_auc:.4f}")

    # ── Optimal threshold from fold's validation set ──
    model.eval()
    val_y_true, val_y_probs = [], []
    with torch.no_grad():
        for imgs, labels in fold_val_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True).float()
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    logits = model(imgs)
            else:
                logits = model(imgs)
            probs = torch.sigmoid(logits)
            val_y_true.extend(labels.long().cpu().numpy())
            val_y_probs.extend(probs.cpu().numpy())

    val_y_true = np.array(val_y_true)
    val_y_probs = np.array(val_y_probs)
    opt_thresh, opt_sens, opt_spec = find_optimal_threshold(
        val_y_true, val_y_probs, min_sensitivity=MIN_SENSITIVITY_TARGET
    )
    fold_thresholds.append(opt_thresh)
    print(f"  Optimal threshold (val): {opt_thresh:.4f}  (Sens={opt_sens:.4f}, Spec={opt_spec:.4f})")

    # ── Fold validation metrics ──
    val_auc_roc = roc_auc_score(val_y_true, val_y_probs)
    val_auc_pr = average_precision_score(val_y_true, val_y_probs)
    val_y_pred = (val_y_probs >= opt_thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(val_y_true, val_y_pred).ravel()
    fold_val_results.append({
        'fold': fold_idx + 1,
        'auc_roc': val_auc_roc,
        'auc_pr': val_auc_pr,
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'accuracy': np.mean(val_y_true == val_y_pred),
        'threshold': opt_thresh,
    })

    # ── Evaluate on held-out test set ──
    test_y_true, test_y_probs = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True).float()
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    logits = model(imgs)
            else:
                logits = model(imgs)
            probs = torch.sigmoid(logits)
            test_y_true.extend(labels.long().cpu().numpy())
            test_y_probs.extend(probs.cpu().numpy())

    test_y_true = np.array(test_y_true)
    test_y_probs = np.array(test_y_probs)
    fold_test_probs.append(test_y_probs)

    test_auc_roc = roc_auc_score(test_y_true, test_y_probs)
    test_auc_pr = average_precision_score(test_y_true, test_y_probs)
    test_y_pred = (test_y_probs >= opt_thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(test_y_true, test_y_pred).ravel()
    test_sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    test_spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    test_acc = np.mean(test_y_true == test_y_pred)
    f1 = precision_recall_fscore_support(test_y_true, test_y_pred, average='binary', zero_division=0)[2]
    f2 = fbeta_score(test_y_true, test_y_pred, beta=2, average='binary', zero_division=0)

    fold_test_results.append({
        'fold': fold_idx + 1,
        'auc_roc': test_auc_roc,
        'auc_pr': test_auc_pr,
        'sensitivity': test_sens,
        'specificity': test_spec,
        'accuracy': test_acc,
        'f1': f1,
        'f2': f2,
        'threshold': opt_thresh,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
    })

    print(f"  Test — AUC: {test_auc_roc:.4f} | Sens: {test_sens:.4f} | Spec: {test_spec:.4f} | Acc: {test_acc:.4f}")

    # ── Save best fold model for attention visualization ──
    if fold_idx == 0 or test_auc_roc > max([r['auc_roc'] for r in fold_test_results[:-1]], default=0):
        best_fold_model_path = "temp_best_fold_model_swinv2.pth"
        torch.save({
            'model_state_dict': best_model_state,
            'fold_idx': fold_idx + 1,
            'test_auc': test_auc_roc,
            'threshold': opt_thresh,
        }, best_fold_model_path)
        print(f"  ★ Saved as best fold model so far (test AUC: {test_auc_roc:.4f})")

    # ── Cleanup GPU memory ──
    del model, backbone_copy, optimizer_ft, optimizer_warmup, scheduler_ft, scheduler_warmup
    del criterion, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'='*70}")
print(f"  ALL {K_FOLDS} FOLDS COMPLETE")
print(f"{'='*70}")

## 7. Cross-Validation Results Summary

In [ ]:
df_val = pd.DataFrame(fold_val_results)
df_test = pd.DataFrame(fold_test_results)

print("=" * 80)
print("  STRATIFIED 5-FOLD CROSS-VALIDATION — RESULTS SUMMARY")
print("=" * 80)

# ── Per-fold results table ──
print("\n┌─── Per-Fold Test Set Results ───┐\n")
print(df_test[['fold', 'auc_roc', 'auc_pr', 'sensitivity', 'specificity', 'accuracy', 'f1', 'f2', 'threshold']].to_string(index=False, float_format='{:.4f}'.format))

# ── Aggregated metrics ──
metrics_to_report = ['auc_roc', 'auc_pr', 'sensitivity', 'specificity', 'accuracy', 'f1', 'f2']
print(f"\n┌─── Aggregated Test Metrics (mean ± std) ───┐\n")
for metric in metrics_to_report:
    vals = df_test[metric].values
    print(f"  {metric:<16s}: {vals.mean():.4f} ± {vals.std():.4f}  (range: {vals.min():.4f} – {vals.max():.4f})")

print(f"\n  Threshold (mean): {df_test['threshold'].mean():.4f} ± {df_test['threshold'].std():.4f}")

# ── Aggregated confusion matrix (sum across folds) ──
total_tp = df_test['tp'].sum()
total_tn = df_test['tn'].sum()
total_fp = df_test['fp'].sum()
total_fn = df_test['fn'].sum()
print(f"\n┌─── Aggregated Confusion Matrix (summed over {K_FOLDS} folds) ───┐")
print(f"  TN={total_tn:<5d} FP={total_fp:<5d}")
print(f"  FN={total_fn:<5d} TP={total_tp:<5d}")

# ── Validation set summary (sanity check) ──
print(f"\n┌─── Per-Fold Validation Set Results ───┐\n")
print(df_val[['fold', 'auc_roc', 'auc_pr', 'sensitivity', 'specificity', 'accuracy', 'threshold']].to_string(index=False, float_format='{:.4f}'.format))

print(f"\n  Val AUC-ROC (mean ± std): {df_val['auc_roc'].mean():.4f} ± {df_val['auc_roc'].std():.4f}")
print(f"  Val Sensitivity (mean ± std): {df_val['sensitivity'].mean():.4f} ± {df_val['sensitivity'].std():.4f}")
print("=" * 80)

## 8. Cross-Validation Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('SwinV2-Tiny — 5-Fold CV Training Curves', fontsize=14, fontweight='bold')

# Colors for each fold
colors = plt.cm.tab10(np.linspace(0, 1, K_FOLDS))

# Training loss
ax = axes[0, 0]
for i, hist in enumerate(fold_histories):
    ax.plot(hist['train_loss'], color=colors[i], label=f'Fold {i+1}')
ax.set_title('Training Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Validation loss
ax = axes[0, 1]
for i, hist in enumerate(fold_histories):
    ax.plot(hist['val_loss'], color=colors[i], label=f'Fold {i+1}')
ax.set_title('Validation Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Validation AUC
ax = axes[0, 2]
for i, hist in enumerate(fold_histories):
    ax.plot(hist['val_auc'], color=colors[i], label=f'Fold {i+1}')
ax.set_title('Validation AUC-ROC')
ax.set_xlabel('Epoch')
ax.set_ylabel('AUC')
ax.legend()
ax.grid(True, alpha=0.3)

# Training accuracy
ax = axes[1, 0]
for i, hist in enumerate(fold_histories):
    ax.plot(hist['train_acc'], color=colors[i], label=f'Fold {i+1}')
ax.set_title('Training Accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.legend()
ax.grid(True, alpha=0.3)

# Validation sensitivity
ax = axes[1, 1]
for i, hist in enumerate(fold_histories):
    ax.plot(hist['val_sens'], color=colors[i], label=f'Fold {i+1}')
ax.set_title('Validation Sensitivity')
ax.set_xlabel('Epoch')
ax.set_ylabel('Sensitivity')
ax.legend()
ax.grid(True, alpha=0.3)

# Validation specificity
ax = axes[1, 2]
for i, hist in enumerate(fold_histories):
    ax.plot(hist['val_spec'], color=colors[i], label=f'Fold {i+1}')
ax.set_title('Validation Specificity')
ax.set_xlabel('Epoch')
ax.set_ylabel('Specificity')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Ensemble Evaluation on Test Set

In [ ]:
# Ensemble by averaging probabilities across folds
ensemble_probs = np.mean(fold_test_probs, axis=0)

# Use mean threshold from folds
ensemble_threshold = np.mean(fold_thresholds)

# Find optimal threshold for ensemble
opt_thresh_ens, opt_sens_ens, opt_spec_ens = find_optimal_threshold(
    test_y_true, ensemble_probs, min_sensitivity=MIN_SENSITIVITY_TARGET
)

print(f"\n{'='*70}")
print(f"  ENSEMBLE EVALUATION (averaging {K_FOLDS} folds' probabilities)")
print(f"{'='*70}")

# Compute ensemble metrics
ensemble_auc_roc = roc_auc_score(test_y_true, ensemble_probs)
ensemble_auc_pr = average_precision_score(test_y_true, ensemble_probs)

ensemble_preds = (ensemble_probs >= opt_thresh_ens).astype(int)
tn, fp, fn, tp = confusion_matrix(test_y_true, ensemble_preds).ravel()
ensemble_sens = tp / (tp + fn) if (tp + fn) > 0 else 0
ensemble_spec = tn / (tn + fp) if (tn + fp) > 0 else 0
ensemble_acc = np.mean(test_y_true == ensemble_preds)
ensemble_f1 = precision_recall_fscore_support(test_y_true, ensemble_preds, average='binary', zero_division=0)[2]
ensemble_f2 = fbeta_score(test_y_true, ensemble_preds, beta=2, average='binary', zero_division=0)

print(f"\n  Optimal threshold: {opt_thresh_ens:.4f}")
print(f"\n  AUC-ROC:     {ensemble_auc_roc:.4f}")
print(f"  AUC-PR:      {ensemble_auc_pr:.4f}")
print(f"  Sensitivity: {ensemble_sens:.4f}")
print(f"  Specificity: {ensemble_spec:.4f}")
print(f"  Accuracy:    {ensemble_acc:.4f}")
print(f"  F1:          {ensemble_f1:.4f}")
print(f"  F2:          {ensemble_f2:.4f}")

print(f"\n  Confusion Matrix:")
print(f"    TN={tn}  FP={fp}")
print(f"    FN={fn}  TP={tp}")
print(f"{'='*70}")

## 10. ROC and Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
ax = axes[0]
fpr, tpr, _ = roc_curve(test_y_true, ensemble_probs)
ax.plot(fpr, tpr, color='blue', lw=2, label=f'Ensemble (AUC = {ensemble_auc_roc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — SwinV2-Tiny Ensemble')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

# Precision-Recall Curve
ax = axes[1]
precision, recall, _ = precision_recall_curve(test_y_true, ensemble_probs)
ax.plot(recall, precision, color='green', lw=2, label=f'Ensemble (AP = {ensemble_auc_pr:.4f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — SwinV2-Tiny Ensemble')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Attention Visualization (Gradient-Based Saliency)

In [ ]:
def get_saliency_map(model, image_tensor):
    """Generate gradient-based saliency map for SwinV2."""
    image_tensor = image_tensor.unsqueeze(0).to(DEVICE)
    image_tensor.requires_grad_(True)
    try:
        with torch.enable_grad():
            outputs = model.backbone.swinv2(image_tensor)
            features = outputs.last_hidden_state.mean(dim=1)
            saliency = features.sum()
            saliency.backward()
        gradients = image_tensor.grad.data.abs()
        saliency_map = gradients[0].mean(dim=0).cpu().numpy()
        # Resize to a standard grid size
        saliency_resized = cv2.resize(saliency_map, (16, 16))
        saliency_resized = (saliency_resized - saliency_resized.min()) / (saliency_resized.max() - saliency_resized.min() + 1e-6)
        return saliency_resized.astype(np.float32)
    except Exception as e:
        return None


def overlay_attention_on_image(image_np, attention_map, alpha=0.5):
    """Overlay attention heatmap on image."""
    h, w = image_np.shape[:2]
    attn_upsampled = cv2.resize(attention_map, (w, h))
    cmap = plt.get_cmap('jet')
    heatmap_rgb = cmap(attn_upsampled)[:, :, :3]
    heatmap_rgb = (heatmap_rgb * 255).astype(np.uint8)
    image_float = image_np.astype(np.float32) / 255.0
    heatmap_float = heatmap_rgb.astype(np.float32) / 255.0
    blended = (alpha * heatmap_float + (1 - alpha) * image_float)
    blended = np.clip(blended * 255, 0, 255).astype(np.uint8)
    return blended


def visualize_attention_quadrant(quadrant_name, quadrant_indices, model,
                                 class_name_true, class_name_pred, threshold=0.5, n_samples=3):
    """Visualize attention for SwinV2 model."""
    if len(quadrant_indices) == 0:
        print(f"\n  No {quadrant_name} samples.")
        return
    sample_indices = np.random.choice(quadrant_indices, size=min(n_samples, len(quadrant_indices)), replace=False)
    fig, axes = plt.subplots(n_samples, 3, figsize=(15, 5*n_samples))
    if n_samples == 1:
        axes = axes.reshape(1, 3)
    fig.suptitle(f"{quadrant_name}\nTrue: {class_name_true} | Pred: {class_name_pred}",
                 fontsize=14, fontweight='bold')
    for row, test_idx in enumerate(sample_indices):
        row_data = test_manifest.iloc[test_idx]
        cancer_type = row_data['cancer_type']
        prob = ensemble_probs[test_idx]
        filenames = str(row_data['dscope_files']).split(';')
        dscope_path = os.path.join(IMAGES_DIR, filenames[0]) if filenames else None
        try:
            dscope_img = Image.open(dscope_path).convert('RGB')
            dscope_orig = np.array(dscope_img.resize((256, 256)))
            dscope_tensor = eval_transform(dscope_img)
            attn_dscope = get_saliency_map(model, dscope_tensor)
            if attn_dscope is not None:
                dscope_with_attn = overlay_attention_on_image(dscope_orig, attn_dscope, alpha=0.4)
            else:
                dscope_with_attn = dscope_orig
        except Exception as e:
            dscope_orig = np.zeros((256, 256, 3), dtype=np.uint8)
            dscope_with_attn = dscope_orig
            attn_dscope = np.zeros((16, 16))
        
        # Original image
        axes[row, 0].imshow(dscope_orig)
        axes[row, 0].set_title(f"Original\n{cancer_type}")
        axes[row, 0].axis('off')
        
        # Saliency map
        if attn_dscope is not None:
            axes[row, 1].imshow(attn_dscope, cmap='jet')
        axes[row, 1].set_title(f"Saliency Map\nProb: {prob:.3f}")
        axes[row, 1].axis('off')
        
        # Overlay
        axes[row, 2].imshow(dscope_with_attn)
        axes[row, 2].set_title("Overlay")
        axes[row, 2].axis('off')
    
    plt.tight_layout()
    plt.show()


print("✓ Attention visualization utilities defined.")

In [ ]:
# Load best fold model for visualization
checkpoint = torch.load("temp_best_fold_model_swinv2.pth", map_location=DEVICE)
print(f"Loading best fold model (Fold {checkpoint['fold_idx']}, Test AUC: {checkpoint['test_auc']:.4f})")

# Recreate model
viz_backbone = Swinv2ForImageClassification.from_pretrained(
    "microsoft/swinv2-tiny-patch4-window8-256",
    num_labels=1,
    ignore_mismatched_sizes=True
)
viz_model = BinarySwinV2Model(
    backbone=viz_backbone, embed_dim=embed_dim, hidden=512, dropout=0.3
).to(DEVICE)
viz_model.load_state_dict(checkpoint['model_state_dict'])
viz_model.eval()

# Get confusion matrix indices
viz_threshold = checkpoint['threshold']
viz_preds = (ensemble_probs >= viz_threshold).astype(int)

tp_idx = np.where((test_y_true == 1) & (viz_preds == 1))[0]
tn_idx = np.where((test_y_true == 0) & (viz_preds == 0))[0]
fp_idx = np.where((test_y_true == 0) & (viz_preds == 1))[0]
fn_idx = np.where((test_y_true == 1) & (viz_preds == 0))[0]

print(f"\nConfusion matrix breakdown:")
print(f"  TP: {len(tp_idx)} | TN: {len(tn_idx)} | FP: {len(fp_idx)} | FN: {len(fn_idx)}")

In [ ]:
# Visualize True Positives
visualize_attention_quadrant("TRUE POSITIVES", tp_idx, viz_model,
                             "Malignant", "Malignant", viz_threshold, n_samples=3)

In [ ]:
# Visualize True Negatives
visualize_attention_quadrant("TRUE NEGATIVES", tn_idx, viz_model,
                             "Benign", "Benign", viz_threshold, n_samples=3)

In [ ]:
# Visualize False Positives
visualize_attention_quadrant("FALSE POSITIVES", fp_idx, viz_model,
                             "Benign", "Malignant", viz_threshold, n_samples=3)

In [ ]:
# Visualize False Negatives
visualize_attention_quadrant("FALSE NEGATIVES", fn_idx, viz_model,
                             "Malignant", "Benign", viz_threshold, n_samples=3)

## 12. Cleanup

In [ ]:
# Remove temporary model file
import os
if os.path.exists("temp_best_fold_model_swinv2.pth"):
    os.remove("temp_best_fold_model_swinv2.pth")
    print("✓ Temporary model file removed.")

print("\n" + "="*70)
print("  EXPERIMENT COMPLETE")
print("="*70)